In [1]:
!pip install --upgrade scikit-learn
!pip install category_encoders
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 65.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


# 1. Imports & Configuration

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import warnings
import os
import gc
import torch

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.cluster import KMeans

from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error 

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.preprocessing import TargetEncoder

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer

# from category_encoders import TargetEncoder

from joblib import Parallel, delayed

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
RANDOM_STATE = 42
SEEDS = [42, 2024, 123]

# 2. Data Loading & Integration

In [3]:
train_df = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')
test_df = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')
original_df = pd.read_csv('/kaggle/input/exam-score-prediction-dataset/Exam_Score_Prediction.csv')

if 'id' in original_df.columns:
    original_df = original_df.drop(columns=['id'])

TARGET = 'exam_score'
CATS = train_df.select_dtypes('object').columns.to_list() 

# 3. Feature Engineering

In [4]:
def preprocess(df):
    df = df.copy()
    eps = 1e-5
    
    # 1. Base Numeric Transforms
    df['study_hours_sq'] = df['study_hours'] ** 2
    df['attendance_sq'] = df['class_attendance'] ** 2
    df['sleep_hours_sq'] = df['sleep_hours'] ** 2
    df['age_sq'] = df['age'] ** 2
    
    df['log_study'] = np.log1p(df['study_hours'])
    df['log_attendance'] = np.log1p(df['class_attendance'])
    df['log_sleep'] = np.log1p(df['sleep_hours'])
    
    df['sqrt_study'] = np.sqrt(df['study_hours'])
    df['sqrt_attendance'] = np.sqrt(df['class_attendance'])
    
    # 2. Numeric Interactions
    df['study_x_attend'] = df['study_hours'] * df['class_attendance']
    df['study_x_sleep'] = df['study_hours'] * df['sleep_hours']
    df['attend_x_sleep'] = df['class_attendance'] * df['sleep_hours']
    df['age_x_study'] = df['age'] * df['study_hours']
    
    # 3. Ratios
    df['study_over_sleep'] = df['study_hours'] / (df['sleep_hours'] + eps)
    df['attend_over_sleep'] = df['class_attendance'] / (df['sleep_hours'] + eps)
    df['attend_over_study'] = df['class_attendance'] / (df['study_hours'] + eps)
    df['efficiency'] = (df['study_hours'] * df['class_attendance']) / (df['sleep_hours'] + 1)
    
    # 4. Ordinal Mappings
    sleep_map = {'Poor': 0, 'Average': 1, 'Good': 2, 'poor': 0, 'average': 1, 'good': 2}
    facility_map = {'Low': 0, 'Moderate': 1, 'High': 2, 'low': 0, 'moderate': 1, 'high': 2}
    difficulty_map = {'Easy': 0, 'Moderate': 1, 'Hard': 2, 'easy': 0, 'moderate': 1, 'hard': 2}
    
    df['sleep_qual_num'] = df['sleep_quality'].map(sleep_map).fillna(1)
    df['facility_num'] = df['facility_rating'].map(facility_map).fillna(1)
    df['difficulty_num'] = df['exam_difficulty'].map(difficulty_map).fillna(1)
    
    # 5. Ordinal Interactions
    df['study_x_sleep_qual'] = df['study_hours'] * df['sleep_qual_num']
    df['attend_x_facility'] = df['class_attendance'] * df['facility_num']
    df['sleep_x_difficulty'] = df['sleep_hours'] * df['difficulty_num']
    df['facility_x_sleep_qual'] = df['facility_num'] * df['sleep_qual_num']
    df['difficulty_x_facility'] = df['difficulty_num'] * df['facility_num']
    
    # 6. Bins
    df["study_bin"] = pd.cut(df["study_hours"], bins=[-1, 2, 4, 6, 8, 100], labels=[0, 1, 2, 3, 4]).astype(float)
    df["attendance_bin"] = pd.cut(df["class_attendance"], bins=[-1, 60, 75, 85, 95, 101], labels=[0, 1, 2, 3, 4]).astype(float)
    df["sleep_bin"] = pd.cut(df["sleep_hours"], bins=[-1, 5, 6, 7, 8, 100], labels=[0, 1, 2, 3, 4]).astype(float)
    df["age_bin"] = pd.cut(df["age"], bins=[0, 17, 19, 21, 23, 100], labels=[0, 1, 2, 3, 4]).astype(float)
    
    # 7. Flags & Gaps
    df["high_att_high_study"] = ((df["class_attendance"] >= 90) & (df["study_hours"] >= 6)).astype(int)
    df["ideal_sleep"] = ((df["sleep_hours"] >= 7) & (df["sleep_hours"] <= 9)).astype(int)
    df["high_study_flag"] = (df["study_hours"] >= 7).astype(int)
    
    df['sleep_gap_8'] = (df['sleep_hours'] - 8.0).abs()
    df['attendance_gap_100'] = (df['class_attendance'] - 100.0).abs()

    return df

X_train = preprocess(train_df.drop(columns=[TARGET, 'id'], errors='ignore'))
y_train = train_df[TARGET]
X_test = preprocess(test_df.drop(columns=['id'], errors='ignore'))
X_orig = preprocess(original_df.drop(columns=[TARGET, 'id'], errors='ignore'))
y_orig = original_df[TARGET]

print(f"Feature Set Created. Train Shape: {X_train.shape}")

Feature Set Created. Train Shape: (630000, 45)


# 4. Target Encoding & Ridge Stacking Layer

In [5]:
print("Starting Stacking Layer")

common_cols = X_train.columns.intersection(X_orig.columns).intersection(X_test.columns)
X_train_aligned = X_train[common_cols].copy()
X_orig_aligned = X_orig[common_cols].copy()
X_test_aligned = X_test[common_cols].copy()

# We select columns that are object or category type
CATS_ALIGNED = X_train_aligned.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Encoding {len(CATS_ALIGNED)} categorical features: {CATS_ALIGNED}")

oof_preds_lr = np.zeros(len(X_train))
test_preds_lr = np.zeros((len(X_test), 10)) 
orig_preds_lr = np.zeros(len(X_orig)) 

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_aligned, y_train)):
    X_tr_fold, y_tr_fold = X_train_aligned.iloc[train_idx], y_train.iloc[train_idx]
    X_val_fold, y_val_fold = X_train_aligned.iloc[val_idx], y_train.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr_fold, X_orig_aligned], axis=0)
    y_tr_aug = pd.concat([y_tr_fold, y_orig], axis=0)
    
    # 3. ENCODING 
    enc = TargetEncoder(smooth='auto', target_type='continuous', random_state=42)
    
    # Fit on Augmented Train
    X_tr_cat_encoded = enc.fit_transform(X_tr_aug[CATS_ALIGNED], y_tr_aug)
    X_val_cat_encoded = enc.transform(X_val_fold[CATS_ALIGNED])
    X_test_cat_encoded = enc.transform(X_test_aligned[CATS_ALIGNED])
    X_orig_cat_encoded = enc.transform(X_orig_aligned[CATS_ALIGNED])
    
    # Replace columns in the dataset
    X_tr_aug_final = X_tr_aug.copy()
    X_tr_aug_final[CATS_ALIGNED] = X_tr_cat_encoded
    
    X_val_final = X_val_fold.copy()
    X_val_final[CATS_ALIGNED] = X_val_cat_encoded
    
    X_test_final = X_test_aligned.copy()
    X_test_final[CATS_ALIGNED] = X_test_cat_encoded
    
    X_orig_final = X_orig_aligned.copy()
    X_orig_final[CATS_ALIGNED] = X_orig_cat_encoded

    # 4. Impute & Clean
    X_tr_aug_final = X_tr_aug_final.astype(float)
    X_val_final = X_val_final.astype(float)
    X_test_final = X_test_final.astype(float)
    X_orig_final = X_orig_final.astype(float)
    
    imputer = SimpleImputer(strategy='mean')
    X_tr_aug_final = imputer.fit_transform(X_tr_aug_final)
    X_val_final = imputer.transform(X_val_final)
    X_test_final = imputer.transform(X_test_final)
    X_orig_final = imputer.transform(X_orig_final)
    
    # 5. Train Ridge
    alphas = np.logspace(-3, 3, 20)
    model = RidgeCV(alphas=alphas, cv=3) 
    model.fit(X_tr_aug_final, y_tr_aug)
    
    oof_preds_lr[val_idx] = np.clip(model.predict(X_val_final), 0, 100)
    test_preds_lr[:, fold] = np.clip(model.predict(X_test_final), 0, 100)
    orig_preds_lr += np.clip(model.predict(X_orig_final), 0, 100) / 10
    
    score = root_mean_squared_error(y_val_fold, oof_preds_lr[val_idx])
    print(f"Fold {fold+1} Ridge RMSE: {score:.4f}")

X_train['feature_lr_pred'] = oof_preds_lr
X_test['feature_lr_pred'] = test_preds_lr.mean(axis=1)
X_orig['feature_lr_pred'] = orig_preds_lr

print("Stacking Complete.")

Starting Stacking Layer
Encoding 7 categorical features: ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']
Fold 1 Ridge RMSE: 8.8611
Fold 2 Ridge RMSE: 8.9094
Fold 3 Ridge RMSE: 8.8614
Fold 4 Ridge RMSE: 8.9192
Fold 5 Ridge RMSE: 8.8731
Fold 6 Ridge RMSE: 8.8917
Fold 7 Ridge RMSE: 8.9056
Fold 8 Ridge RMSE: 8.8782
Fold 9 Ridge RMSE: 8.9015
Fold 10 Ridge RMSE: 8.9235
Stacking Complete.


# 5. Training: XGBoost
Defining the XGBoost training logic. 
* **Seed Averaging:** Inside every fold, we train multiple models with different random seeds to reduce variance.
* **Hyperparameters:** Optimized for depth, learning rate, and regularization (Reg Alpha/Lambda).

In [6]:
print("Starting XGBoost Seed Averaging")

common_cols = X_train.columns.intersection(X_orig.columns).intersection(X_test.columns)
print(f"Features: {len(common_cols)} columns")

X_train_final = X_train[common_cols].copy()
X_orig_final = X_orig[common_cols].copy()
X_test_final = X_test[common_cols].copy()
y_train_final = y_train.copy()
y_orig_final = y_orig.copy()

CATS_FINAL = [c for c in CATS if c in common_cols]
for col in CATS_FINAL:
    X_train_final[col] = X_train_final[col].astype('category')
    X_orig_final[col] = X_orig_final[col].astype('category')
    X_test_final[col] = X_test_final[col].astype('category')

Starting XGBoost Seed Averaging
Features: 46 columns


In [7]:
def train_fold_final_seeded(fold, train_idx, val_idx):
    device_id = f"cuda:{fold % 2}"
    
    X_tr_fold, y_tr_fold = X_train_final.iloc[train_idx], y_train_final.iloc[train_idx]
    X_val_fold, y_val_fold = X_train_final.iloc[val_idx], y_train_final.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr_fold, X_orig_final], axis=0)
    y_tr_aug = pd.concat([y_tr_fold, y_orig_final], axis=0)
    
    val_preds_avg = np.zeros(len(X_val_fold))
    test_preds_avg = np.zeros(len(X_test_final))
    
    for seed in SEEDS:
        params = {
            'n_estimators': 20000,
            'learning_rate': 0.004,
            'max_depth': 9,
            'subsample': 0.78,
            'colsample_bytree': 0.55,
            'colsample_bynode': 0.65,
            'reg_lambda': 6,
            'reg_alpha': 0.15,
            'min_child_weight': 6,
            'tree_method': 'hist',
            'enable_categorical': True,
            'early_stopping_rounds': 100,
            'device': device_id,
            'random_state': seed,
            'n_jobs': 4,
            'verbosity': 0,
            'eval_metric': 'rmse'
        }
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_tr_aug, y_tr_aug, eval_set=[(X_val_fold, y_val_fold)], verbose=False)
        
        val_preds_avg += model.predict(X_val_fold) / len(SEEDS)
        test_preds_avg += model.predict(X_test_final) / len(SEEDS)
        
        del model
        gc.collect()
    
    rmse = root_mean_squared_error(y_val_fold, val_preds_avg)
    print(f"✅ Fold {fold+1} Finished (Avg 3 Seeds) | RMSE: {rmse:.5f}")
    
    return val_idx, val_preds_avg, test_preds_avg

In [8]:
# EXECUTE
results = Parallel(n_jobs=2, backend="loky")(
    delayed(train_fold_final_seeded)(fold, train_idx, val_idx)
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_final, y_train_final))
)

# AGGREGATE
oof_preds_final = np.zeros(len(X_train_final))
test_preds_final = np.zeros(len(X_test_final))

for val_idx, val_pred, test_pred in results:
    oof_preds_final[val_idx] = val_pred
    test_preds_final += test_pred / 10

xgb_oof = oof_preds_final.copy()
xgb_test = test_preds_final.copy()

global_rmse = root_mean_squared_error(y_train_final, oof_preds_final)

# Saving XGBoost Outputs
xgb_oof = oof_preds_final.copy()
xgb_test = test_preds_final.copy()

print(f"\n XGBoost Seed Averaging Complete.")
print(f"Global OOF RMSE: {global_rmse:.5f}")

✅ Fold 1 Finished (Avg 3 Seeds) | RMSE: 8.68859
✅ Fold 3 Finished (Avg 3 Seeds) | RMSE: 8.68643
✅ Fold 2 Finished (Avg 3 Seeds) | RMSE: 8.74891
✅ Fold 4 Finished (Avg 3 Seeds) | RMSE: 8.74853
✅ Fold 5 Finished (Avg 3 Seeds) | RMSE: 8.70275
✅ Fold 6 Finished (Avg 3 Seeds) | RMSE: 8.72189
✅ Fold 8 Finished (Avg 3 Seeds) | RMSE: 8.71444
✅ Fold 10 Finished (Avg 3 Seeds) | RMSE: 8.76344

 XGBoost Seed Averaging Complete.
Global OOF RMSE: 8.72596


# 6. Training: LightGBM 

In [9]:
print("Starting LightGBM Seed Averaging")

lgbm_oof = np.zeros(len(X_train_final))
lgbm_test = np.zeros(len(X_test_final))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_final, y_train_final)):
    gc.collect()
    print(f"\n Fold {fold+1} Starting...", flush=True)
    
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_final.iloc[train_idx]
    X_val, y_val = X_train_final.iloc[val_idx], y_train_final.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr, X_orig_final], axis=0)
    y_tr_aug = pd.concat([y_tr, y_orig_final], axis=0)
    
    # Temp arrays for averaging seeds within this fold
    val_preds_fold_avg = np.zeros(len(X_val))
    test_preds_fold_avg = np.zeros(len(X_test_final))
    
    for seed in SEEDS:
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'learning_rate': 0.008,
            'n_estimators': 12000,
            'num_leaves': 100,
            'min_child_samples': 40,
            'colsample_bytree': 0.6,
            'subsample': 0.8,
            'subsample_freq': 1,
            'reg_alpha': 2,
            'reg_lambda': 5,
            'random_state': seed,
            'verbosity': -1,
            'n_jobs': 4,
            'device': 'gpu',
            'gpu_device_id': 0,
            'gpu_platform_id': 0,
            'force_col_wise': True
        }
        
        model = lgb.LGBMRegressor(**params)
        
        model.fit(
            X_tr_aug, y_tr_aug,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        
        val_preds_fold_avg += model.predict(X_val) / len(SEEDS)
        test_preds_fold_avg += model.predict(X_test_final) / len(SEEDS)
        
        del model
    
    lgbm_oof[val_idx] = val_preds_fold_avg
    lgbm_test += test_preds_fold_avg / 10
    
    rmse = root_mean_squared_error(y_val, val_preds_fold_avg)
    print(f"✅ Fold {fold+1} Finished (Avg 3 Seeds) | RMSE: {rmse:.5f}")
    
    del X_tr_aug, y_tr_aug, X_tr, X_val
    gc.collect()

print(f"\n LightGBM Seed Averaging Complete.")
print(f"Global LightGBM RMSE: {root_mean_squared_error(y_train_final, lgbm_oof):.5f}")

Starting LightGBM Seed Averaging

 Fold 1 Starting...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


✅ Fold 7 Finished (Avg 3 Seeds) | RMSE: 8.75024
✅ Fold 9 Finished (Avg 3 Seeds) | RMSE: 8.73396
✅ Fold 1 Finished (Avg 3 Seeds) | RMSE: 8.68786

 Fold 2 Starting...
✅ Fold 2 Finished (Avg 3 Seeds) | RMSE: 8.74562

 Fold 3 Starting...
✅ Fold 3 Finished (Avg 3 Seeds) | RMSE: 8.68964

 Fold 4 Starting...
✅ Fold 4 Finished (Avg 3 Seeds) | RMSE: 8.74591

 Fold 5 Starting...
✅ Fold 5 Finished (Avg 3 Seeds) | RMSE: 8.70092

 Fold 6 Starting...
✅ Fold 6 Finished (Avg 3 Seeds) | RMSE: 8.72306

 Fold 7 Starting...
✅ Fold 7 Finished (Avg 3 Seeds) | RMSE: 8.74906

 Fold 8 Starting...
✅ Fold 8 Finished (Avg 3 Seeds) | RMSE: 8.70992

 Fold 9 Starting...
✅ Fold 9 Finished (Avg 3 Seeds) | RMSE: 8.73577

 Fold 10 Starting...
✅ Fold 10 Finished (Avg 3 Seeds) | RMSE: 8.75933

 LightGBM Seed Averaging Complete.
Global LightGBM RMSE: 8.72474


# 7. Training: CatBoost

In [10]:
print("Starting CatBoost Seed Averaging")

def train_fold_catboost(fold, train_idx, val_idx):
    gpu_id = str(fold % 2)
    
    print(f"🔄 Fold {fold+1} Starting on GPU {gpu_id}...", flush=True)
    
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_final.iloc[train_idx]
    X_val, y_val = X_train_final.iloc[val_idx], y_train_final.iloc[val_idx]
    
    X_tr_aug = pd.concat([X_tr, X_orig_final], axis=0)
    y_tr_aug = pd.concat([y_tr, y_orig_final], axis=0)
    
    # Creating Pools ONCE per fold (Re-used for all seeds)
    train_pool = cb.Pool(X_tr_aug, y_tr_aug, cat_features=CATS_FINAL)
    val_pool = cb.Pool(X_val, y_val, cat_features=CATS_FINAL)
    test_pool = cb.Pool(X_test_final, cat_features=CATS_FINAL)
    
    # Temp arrays for averaging seeds within this fold
    val_preds_fold_avg = np.zeros(len(X_val))
    test_preds_fold_avg = np.zeros(len(X_test_final))
    
    for seed in SEEDS:
        params = {
            'loss_function': 'RMSE',
            'eval_metric': 'RMSE',
            'learning_rate': 0.02,
            'iterations': 8000,
            'depth': 6,
            'random_strength': 0.5,
            'l2_leaf_reg': 5,
            'task_type': 'GPU',
            'devices': gpu_id,
            'random_seed': seed,
            'verbose': 0,
            'early_stopping_rounds': 100
        }
        
        model = cb.CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=val_pool, verbose=False)
        
        val_preds_fold_avg += model.predict(val_pool) / len(SEEDS)
        test_preds_fold_avg += model.predict(test_pool) / len(SEEDS)
        
        del model
        gc.collect()
    
    rmse = root_mean_squared_error(y_val, val_preds_fold_avg)
    print(f"✅ Fold {fold+1} Finished (Avg 3 Seeds) | RMSE: {rmse:.5f}")
    
    # Cleanup
    del train_pool, val_pool, test_pool, X_tr_aug, y_tr_aug
    gc.collect()
    
    return val_idx, val_preds_fold_avg, test_preds_fold_avg

Starting CatBoost Seed Averaging


In [11]:
# EXECUTE
results = Parallel(n_jobs=2, backend="loky")(
    delayed(train_fold_catboost)(fold, train_idx, val_idx)
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_final, y_train_final))
)

# AGGREGATE 
cat_oof = np.zeros(len(X_train_final))
cat_test = np.zeros(len(X_test_final))

for val_idx, val_pred, test_pred in results:
    cat_oof[val_idx] = val_pred
    cat_test += test_pred / 10

print(f"\n CatBoost Seed Averaging Complete.")
print(f"Global CatBoost RMSE: {root_mean_squared_error(y_train_final, cat_oof):.5f}")

🔄 Fold 1 Starting on GPU 0...
🔄 Fold 2 Starting on GPU 1...
✅ Fold 1 Finished (Avg 3 Seeds) | RMSE: 8.73040
🔄 Fold 3 Starting on GPU 0...
✅ Fold 2 Finished (Avg 3 Seeds) | RMSE: 8.78902
🔄 Fold 4 Starting on GPU 1...
✅ Fold 3 Finished (Avg 3 Seeds) | RMSE: 8.73571
🔄 Fold 5 Starting on GPU 0...
✅ Fold 4 Finished (Avg 3 Seeds) | RMSE: 8.79474
🔄 Fold 6 Starting on GPU 1...
✅ Fold 5 Finished (Avg 3 Seeds) | RMSE: 8.74838
🔄 Fold 7 Starting on GPU 0...
✅ Fold 6 Finished (Avg 3 Seeds) | RMSE: 8.76811
🔄 Fold 8 Starting on GPU 1...
✅ Fold 7 Finished (Avg 3 Seeds) | RMSE: 8.79029
🔄 Fold 9 Starting on GPU 0...
✅ Fold 8 Finished (Avg 3 Seeds) | RMSE: 8.75909
🔄 Fold 10 Starting on GPU 1...

 CatBoost Seed Averaging Complete.
Global CatBoost RMSE: 8.77007


# 8. Automated Ensemble Weighting

In [12]:
X_blend = pd.DataFrame({
    'XGB': xgb_oof,
    'LGBM': lgbm_oof,
    'CAT': cat_oof
})

blender = LinearRegression(positive=True, fit_intercept=False)
blender.fit(X_blend, y_train_final)

weights = blender.coef_
xgb_weight = weights[0]
lgbm_weight = weights[1]
cat_weight = weights[2]

total = xgb_weight + lgbm_weight + cat_weight
xgb_weight /= total
lgbm_weight /= total
cat_weight /= total

print(f"Optimal Weights Found:")
print(f"XGBoost Weight:  {xgb_weight:.4f}")
print(f"LightGBM Weight: {lgbm_weight:.4f}")
print(f"CatBoost Weight: {cat_weight:.4f}")

final_oof_blend = (xgb_weight * xgb_oof) + (lgbm_weight * lgbm_oof) + (cat_weight * cat_oof)
blend_rmse = root_mean_squared_error(y_train_final, final_oof_blend)
print(f"   Combined OOF RMSE: {blend_rmse:.5f}")

Optimal Weights Found:
XGBoost Weight:  0.4650
LightGBM Weight: 0.5350
CatBoost Weight: 0.0000
   Combined OOF RMSE: 8.72101


# 9. Final Submission

In [13]:
final_test_blend = (xgb_weight * xgb_test) + (lgbm_weight * lgbm_test) + (cat_weight * cat_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'exam_score': np.clip(final_test_blend, 0, 100)
})

submission.to_csv('submission.csv', index=False)
print("Submission Saved: 'submission.csv'")

Submission Saved: 'submission.csv'
